## Analisis Exploratorio de los Datos (EDA) 
### Christian Pascual

---

1. **Carga de Librerías**

In [ ]:
# manipulación de datos
import pandas as pd

# visualización
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype, is_bool_dtype
# estilo de gráficos
sns.set(style="whitegrid")

# función para cargar datos
from cargar_datos import cargar_datos

---

2. **Exploración Inicial del Dataset**

In [ ]:
# cargar datos
df = cargar_datos()

# ver primeras filas
df.head()

In [ ]:
# dimensiones del dataset
df.shape

In [ ]:
# nombres de columnas
df.columns

---

3. **Analisis inicial**

In [ ]:
# columnas categóricas, numéricas y target
target = 'Pago_atiempo'
ordinal_col = ['tipo_credito', 'tendencia_ingresos']

def analisis_dataset(df):
    resumen = []
    for col in df.columns:
        uniques = df[col].nunique()
        tipos_var = df[col].dtype
        
        if col == target or is_bool_dtype(df[col]):
            tipo = 'dicotómica'
        elif col in ordinal_col:
            tipo = 'ordinal'
        elif is_datetime64_any_dtype(df[col]):
            tipo = 'temporal'
        elif is_numeric_dtype(df[col]):
            ent = df[col].dropna().mod(1).eq(0).all()
            if ent and uniques <= 20:
                tipo = 'numérica discreta'
            else:
                tipo = 'numérica continua'
        else:
            tipo = 'nominal'
        
        resumen.append({
            'columna': col,
            'dtype': str(tipos_var),
            'tipo_estadístico': tipo,
            'valores_unicos': uniques,
            '%_nulos': round(df[col].isnull().mean()*100, 2)
        })
        
    return pd.DataFrame(resumen)

df_resumen = analisis_dataset(df)
df_resumen

---

4. **Normalización y Tipos de Datos**

In [ ]:
# valores nulos considerados basura
nulos_reales = ['', ' ', '-', 'nan', 'NaN', 'NAN', 'null', 'NULL', 'None', 'none', 'N/A', 'NA', 'n/a']

# reemplazar
df.replace(nulos_reales, np.nan, inplace=True)

# target como bool
df.dropna(subset=[target_col], inplace=True)
df[target_col] = df[target_col].astype(int).astype(bool)

# columnas categóricas
cols_categoricas = ['tipo_credito', 'tipo_laboral', 'tendencia_ingresos']
for col in cols_categoricas:
    if col in df.columns:
        df[col] = df[col].astype('category')

# columnas numéricas
cols_reservadas = cols_categoricas + ['fecha_prestamo', target_col]
cols_numericas = [c for c in df.columns if c not in cols_reservadas]
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# fechas
if 'fecha_prestamo' in df.columns:
    df['fecha_prestamo'] = pd.to_datetime(df['fecha_prestamo'], errors='coerce')

---

5. **Imputación de nulos**

In [ ]:
# categóricas → 'desconocido'
for col in cols_categoricas:
    if col in df.columns and df[col].isnull().any():
        if "Desconocido" not in df[col].cat.categories:
            df[col] = df[col].cat.add_categories(["Desconocido"])
        df[col] = df[col].fillna("Desconocido")

# numéricas → estrategias por tipo
score_cols = ['puntaje_datacredito', 'puntaje']
for col in score_cols:
    if col in df.columns and df[col].isnull().any():
        df[f'{col}_sin_historial'] = df[col].isnull().astype(int)
        df[col] = df[col].fillna(df[col].quantile(0.05))

cols_zero = [
    'saldo_mora', 'saldo_mora_codeudor', 'total_otros_prestamos', 
    'creditos_sectorFinanciero', 'creditos_sectorCooperativo', 
    'creditos_sectorReal', 'huella_consulta', 'cant_creditosvigentes'
]
for col in cols_zero:
    if col in df.columns and df[col].isnull().any():
        df[col] = df[col].fillna(0)

cols_mediana = [
    'salario_cliente', 'promedio_ingresos_datacredito', 
    'saldo_total', 'saldo_principal', 'capital_prestado', 'cuota_pactada'
]
for col in cols_mediana:
    if col in df.columns and df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

cols_moda = ['plazo_meses', 'edad_cliente']
for col in cols_moda:
    if col in df.columns and df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

---

6. **Análisis univariable**

- **Numéricos**

In [ ]:
def eda_univariable_numerico(df, columnas, target=target_col):
    for col in columnas:
        if col not in df.columns: continue
        serie = df[col].dropna()
        if len(serie) == 0: continue
        unique_vals = serie.nunique()
        es_discreta = (unique_vals <= 20) and (serie % 1 == 0).all()
        
        fig, axes = plt.subplots(1,3, figsize=(18,5))
        
        if es_discreta:
            sns.countplot(x=col, data=df, ax=axes[0], order=sorted(df[col].dropna().unique()))
            axes[0].set_title(f"frecuencia: {col}")
        else:
            sns.histplot(serie, bins=30, kde=True, ax=axes[0])
            axes[0].set_title(f"distribución: {col}")
        
        if not es_discreta and (serie>0).all():
            sns.histplot(np.log1p(serie), bins=30, kde=True, ax=axes[1])
            axes[1].set_title(f"log distribución: {col}")
        else:
            axes[1].axis('off')
        
        if target in df.columns:
            sns.boxplot(x=target, y=col, data=df, ax=axes[2])
            axes[2].set_title(f"{col} vs {target}")
        else:
            axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        q1, q3 = serie.quantile([0.25,0.75])
        iqr = q3-q1
        outliers = ((serie<q1-1.5*iqr) | (serie>q3+1.5*iqr)).sum()
        print(f"{col}: skew={serie.skew():.2f}, kurtosis={serie.kurt():.2f}, outliers={outliers} ({outliers/len(serie)*100:.2f}%)")

cols_analisis = [
    'capital_prestado','salario_cliente','puntaje_datacredito','edad_cliente',
    'cuota_pactada','huella_consulta','saldo_mora','saldo_total'
]
eda_univariable_numerico(df, cols_analisis)

- **Categórico**

In [ ]:
categoricas_para_analizar = ['tipo_credito','tipo_laboral','tendencia_ingresos']
for col in categoricas_para_analizar:
    if col not in df.columns: continue
    conteo = df[col].value_counts(dropna=False)
    porcent = (conteo/len(df)*100).round(2)
    display(pd.DataFrame({'conteo':conteo,'porcentaje':porcent}))
    
    plt.figure(figsize=(10,6))
    sns.countplot(x=df[col], order=df[col].value_counts().index)
    plt.xticks(rotation=45)
    plt.title(col)
    plt.tight_layout()
    plt.show()

---

8. **Análisis Bivariable**

- **Numéricas vs Target**

In [ ]:
# numéricas vs target
def eda_bivariado_numerico(df, columnas, target=target_col):
    for col in columnas:
        if col not in df.columns: continue
        data = df[[col,target]].dropna()
        if len(data)==0: continue
        sns.boxplot(x=target,y=col,data=data)
        plt.title(f"{col} vs {target}")
        plt.show()
        grouped = data.groupby(target)[col].agg(['count','mean','median','std']).round(2)
        display(grouped)
        
cols_num_modelo = ['capital_prestado','salario_cliente','puntaje_datacredito','edad_cliente',
                   'cuota_pactada','huella_consulta','saldo_mora','saldo_total','saldo_principal']
eda_bivariado_numerico(df, cols_num_modelo)

- **Categóricas vs Target**

In [ ]:
# categóricas vs target
def eda_bivariado_categorico(df, columnas, target=target_col):
    for col in columnas:
        if col not in df.columns: continue
        data = df[[col,target]].dropna()
        if len(data)==0: continue
        ct_count = pd.crosstab(data[col],data[target])
        ct_pct = pd.crosstab(data[col],data[target],normalize='index')*100
        tabla = ct_count.copy()
        tabla['total'] = ct_count.sum(axis=1)
        tabla['% pago'] = ct_pct[True] if True in ct_pct.columns else 0
        tabla['% no pago'] = ct_pct[False] if False in ct_pct.columns else 0
        display(tabla)
        plt.figure(figsize=(8,6))
        sns.heatmap(ct_pct,annot=True,fmt=".1f",cmap="YlGnBu")
        plt.title(f'tasa de pago (%) por {col}')
        plt.show()

cols_cat_modelo = ['tipo_credito','tipo_laboral','tendencia_ingresos']
eda_bivariado_categorico(df, cols_cat_modelo)

---

9. **Analisis Multivariable**

- **Matriz de correlaciones**

In [ ]:
numericas = ['capital_prestado','plazo_meses','edad_cliente','salario_cliente',
             'total_otros_prestamos','cuota_pactada','puntaje','puntaje_datacredito',
             'cant_creditosvigentes','huella_consulta','saldo_mora','saldo_total',
             'saldo_principal','saldo_mora_codeudor','creditos_sectorFinanciero',
             'creditos_sectorCooperativo','creditos_sectorReal','promedio_ingresos_datacredito']

# matriz de correlación
plt.figure(figsize=(12,10))
sns.heatmap(df[numericas].corr(),annot=True,fmt=".2f",cmap='coolwarm',vmin=-1,vmax=1)
plt.title('correlación variables numéricas')
plt.show()

- **Pairplot**

In [ ]:
# pairplot (muestra pequeña para no sobrecargar)
cols_pair = ['edad_cliente','salario_cliente','puntaje_datacredito','huella_consulta']
sns.pairplot(df[cols_pair+[target_col]].sample(1000), hue=target_col, diag_kind='kde', corner=True)
plt.show()